In [ ]:
from dotenv import load_dotenv
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.prompts import ChatPromptTemplate
from langchain_classic.storage import LocalFileStore
from langchain_classic.vectorstores import FAISS
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_unstructured import UnstructuredLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores.utils import filter_complex_metadata

load_dotenv()

llm = ChatOpenAI(
  temperature=0.1,
)

# 1. load and transform documents
splitter = CharacterTextSplitter.from_tiktoken_encoder(
  separator="\n",
  chunk_size=600, # 600자 청크로 분할
  chunk_overlap=100, # 청크 간 100자 겹침으로 문맥 유지
)

loader = UnstructuredLoader("./files/How_Neflix_Uses_Java_-_2026_.pdf")

docs = filter_complex_metadata(loader.load_and_split(text_splitter=splitter))


# 2. embeddings with caching
cache_dir = LocalFileStore("./.cache/")

embeddings = OpenAIEmbeddings()

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)


# 3. store in vector database
vectorstore = FAISS.from_documents(docs, cached_embeddings)

# 임베딩 작업을 할 때 먼저 캐시에 embeddings가 있는지 확인하고, 없으면 OpenAI API를 호출하여 임베딩을 생성한 후 캐시에 저장한다. 
# 이렇게 하면 동일한 텍스트에 대해 여러 번 임베딩을 생성할 때 API 호출을 줄일 수 있다.

# 4. create retriever and Stuff LCEL chain
retriever = vectorstore.as_retriever()

prompt = ChatPromptTemplate.from_messages([
  ("system", "You are a helpful assistant. Answer questions using only the following context. If you don't know the answer just say you don't know, don't make it up:\n{context}"),
  ("human", "{question}")
])

chain = (
  {
    "context": retriever, 
    # retriever를 "context" 키에 매핑하여 프롬프트에서 {context}로 사용할 수 있도록 한다. retriever는 질문에 대한 관련 문서를 검색하여 프롬프트에 포함시킨다.
    "question": RunnablePassthrough(), 
    # 질문을 그대로 LLM에게 전달하기 위한 RunnablePassthrough 사용
  }
  | prompt 
  | llm
)

query = "What is the main topic of the document?"

chain.invoke(query)
# 질문(query)을 retriever에 전달하여 관련 문서를 검색하고, 검색된 문서를 프롬프트에 포함시켜 LLM에게 질문을 전달한다. 
# LLM은 프롬프트에 포함된 문맥을 바탕으로 질문에 답변한다.

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


AIMessage(content='The main topic of the document is "How Netflix Uses Java."', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 724, 'total_tokens': 737, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DgP7YbR0GmyH3qHiQD6ikifpseeyX', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e3496-0fe0-7793-bae1-38983f6b8838-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 724, 'output_tokens': 13, 'total_tokens': 737, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})